# Reproducible retrieval of *Staphylococcus aureus* assemblies associated with CCG-UNAM (GenBank-based workflow)

**Author:** Hayde Saracho  
**Institution:** Centro de Ciencias Genómicas, UNAM  

---

## Introduction

*Staphylococcus aureus* is a gram-positive bacterium and a leading cause of both community-acquired and hospital-associated infections worldwide. Its ability to acquire antibiotic resistance — including methicillin-resistant strains (MRSA) — makes it a major public health concern and an active subject of genomic research.

The Centro de Ciencias Genómicas (CCG-UNAM) has sequenced and deposited a collection of *S. aureus* genome assemblies in NCBI GenBank. This notebook provides a fully reproducible workflow to identify and download those assemblies programmatically.

### Why GenBank and not RefSeq?

NCBI maintains two genome databases: **RefSeq**, which contains curated, non-redundant reference sequences, and **GenBank**, which contains all submitted assemblies including those not yet selected for RefSeq. The CCG-UNAM *S. aureus* assemblies are present in GenBank but not in RefSeq, so this workflow queries GenBank's `assembly_summary.txt` directly rather than using the Entrez API, which can return incomplete metadata for GenBank-only records.

### Workflow summary

1. Download the NCBI GenBank bacterial `assembly_summary.txt`
2. Filter for *Staphylococcus aureus* records
3. Identify records submitted by CCG-UNAM using institutional keywords
4. Download and decompress the matching genome FASTA files
5. Verify and report the final dataset

### Runtime and storage estimates

| Step | Estimated time | Disk space |
|---|---|---|
| Download `assembly_summary.txt` | 5–15 min | ~1 GB |
| Filter metadata | < 1 min | negligible |
| Download 108 genomes | 20–60 min | ~300 MB |

Times will vary depending on your internet connection and NCBI server load. The `assembly_summary.txt` file is only downloaded once — if it already exists locally, this step is skipped automatically.

## Dependencies

This notebook requires the following Python packages:

- `pandas` — for reading and filtering the metadata table
- `requests` — for downloading files over HTTPS (no FTP client needed)
- `gzip` and `shutil` — for decompressing downloaded genome files (both are part of the Python standard library)

Install non-standard dependencies with:
```
pip install pandas requests
```

In [1]:
from pathlib import Path
import pandas as pd
import requests
import gzip
import shutil

print("Imports OK.")

Imports OK.


## Parameters

All user-defined parameters are set here. If you want to adapt this workflow for a different organism or institution, this is the only section you need to modify.

- `ORGANISM` — the organism name prefix used to filter `assembly_summary`.
- `INSTITUTION_KEYWORDS` — a list of strings used to identify CCG-UNAM records in the submitter field. Multiple variants are included because institutional names are not standardized across NCBI records.
- `PROJECT_DIR` — set to the parent of the current working directory, assuming the notebook is inside a `notebooks/` subfolder.

In [2]:
ORGANISM = "Staphylococcus aureus"

INSTITUTION_KEYWORDS = [
    "Centro de Ciencias Genomicas",
    "Centro de Ciencias Gen\u00f3micas",
    "UNAM",
]

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR  = PROJECT_DIR / "output"
DATA_DIR    = OUTPUT_DIR / "data"
GENOMES_DIR = OUTPUT_DIR / "genomes"

for d in [OUTPUT_DIR, DATA_DIR, GENOMES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("Directories created:", [str(d) for d in [OUTPUT_DIR, DATA_DIR, GENOMES_DIR]])

PROJECT_DIR: c:\Users\hayde\Downloads\ncbi-ccg-unam-s-aureus-retrieval
Directories created: ['c:\\Users\\hayde\\Downloads\\ncbi-ccg-unam-s-aureus-retrieval\\output', 'c:\\Users\\hayde\\Downloads\\ncbi-ccg-unam-s-aureus-retrieval\\output\\data', 'c:\\Users\\hayde\\Downloads\\ncbi-ccg-unam-s-aureus-retrieval\\output\\genomes']


## Step 1 — Download `assembly_summary` (GenBank)

`assembly_summary.txt` is a tab-separated file maintained by NCBI that contains metadata for all bacterial genome assemblies in GenBank, including accession numbers, organism names, submitting institutions, assembly status, and FTP download paths.

The file is approximately 1 GB and may take 5–15 minutes to download depending on your connection. It is only downloaded once — if the file already exists locally, this step is skipped automatically.

> **Note:** NCBI's FTP server is fully accessible over HTTPS, so no FTP client is required.

In [ ]:
# Note: NCBI FTP is mirrored over HTTPS — no FTP client needed.
SUMMARY_URL = (
    "https://ftp.ncbi.nlm.nih.gov/genomes/genbank/bacteria/assembly_summary.txt"
)
asm_path = DATA_DIR / "assembly_summary.txt"

if asm_path.exists():
    print(f"assembly_summary already exists: {asm_path}")
else:
    print("Downloading assembly_summary.txt ...")
    r = requests.get(SUMMARY_URL, stream=True, timeout=120)
    r.raise_for_status()
    with open(asm_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
    print(f"Saved to: {asm_path}")

Saved to: c:\Users\hayde\Downloads\ncbi-ccg-unam-s-aureus-retrieval\output\data\assembly_summary.txt


## Step 2 — Read metadata

The file has a comment line starting with `#` before the actual header, so `skiprows=1` is used to skip it. The `low_memory=False` option prevents pandas from raising dtype warnings on this large mixed-type file.

In [4]:
df = pd.read_csv(asm_path, sep="\t", skiprows=1, low_memory=False)
print(f"Total rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head(2)

Total rows: 3,026,217  |  Columns: 38


,#assembly_accession,bioproject,biosample,wgs_master,refseq_category,taxid,species_taxid,organism_name,infraspecific_name,isolate,...,replicon_count,scaffold_count,contig_count,annotation_provider,annotation_name,annotation_date,total_gene_count,protein_coding_gene_count,non_coding_gene_count,pubmed_id
0,GCA_036600855.1,PRJNA1051366,SAMN38772065,JBAFXD000000000.1,na,7,7,Azorhizobium caulinodans,strain=CNM20190194,na,...,0,171,171,na,na,NaN,0,0,0,na
1,GCA_036600875.1,PRJNA1051366,SAMN38772066,JBAFXC000000000.1,na,7,7,Azorhizobium caulinodans,strain=CNM20190462,na,...,0,171,171,na,na,NaN,0,0,0,na


## Step 3 — Filter for *Staphylococcus aureus*

The full `assembly_summary.txt` covers all bacteria in GenBank. Here, records are filtered to retain only those whose `organism_name` starts with `"Staphylococcus aureus"`, which also captures subspecies and strain-level designations (e.g., *Staphylococcus aureus* subsp. *aureus*).

In [5]:
sa_df = df[
    df["organism_name"].fillna("").str.startswith(ORGANISM)
].copy()

print(f"S. aureus assemblies: {len(sa_df):,}")

S. aureus assemblies: 122,614


*S. aureus* is one of the most sequenced bacterial pathogens in public databases. The 122,614 assemblies retrieved reflect the large volume of genomic data generated by surveillance, outbreak investigations, and research projects worldwide.

## Step 4 — Filter by institution

CCG-UNAM records are identified by searching the `asm_submitter` field for institutional keywords. Because submitter names are free-text and not standardized in NCBI, multiple keyword variants are used to avoid missing records due to spelling differences (e.g., with or without accent marks).

The search is case-insensitive.

In [8]:
pattern = "|".join(k.lower() for k in INSTITUTION_KEYWORDS)

# Use .loc to avoid SettingWithCopyWarning
sa_df.loc[:, "submitter_clean"] = sa_df["asm_submitter"].fillna("").str.lower()

ccg_df = sa_df[
    sa_df["submitter_clean"].str.contains(pattern, na=False, regex=True)
].copy()

print(f"CCG-UNAM assemblies: {len(ccg_df)}")

CCG-UNAM assemblies: 108


## Inspect detected submitters

The unique submitter names found in the filtered records are listed below. This step confirms that the keyword filter captured the correct institution and helps detect any unexpected matches or name variants.

In [10]:
sorted(ccg_df["asm_submitter"].dropna().unique())

['Centro de Ciencias Genomicas U.N.A.M.']

All 108 records are attributed to a single submitter name, confirming a clean and unambiguous match.

## Step 5 — Build download URLs

The `ftp_path` column in `assembly_summary` contains the directory path for each assembly on the NCBI FTP server. The genomic FASTA file follows a predictable naming convention: `<folder_name>_genomic.fna.gz`.

The `ftp://` protocol is replaced with `https://` so that downloads can be handled directly by `requests` without an FTP client.

In [13]:
def build_url(ftp: str) -> str:
    """Convert an FTP assembly directory path to an HTTPS genomic FASTA URL."""
    if pd.isna(ftp) or str(ftp).strip() in ("", "na"):
        return ""
    https = str(ftp).replace("ftp://", "https://").rstrip("/")
    name  = https.split("/")[-1]
    return f"{https}/{name}_genomic.fna.gz"


ccg_df = ccg_df.copy()           # avoid chained-assignment warnings
ccg_df["url"]  = ccg_df["ftp_path"].apply(build_url)
ccg_df["file"] = ccg_df["#assembly_accession"] + ".fna"

# Report records that lack a download URL
missing = ccg_df["url"].eq("").sum()
if missing:
    print(f"Warning: {missing} record(s) without an FTP path — will be skipped.")

ccg_df[["#assembly_accession", "url"]].head()

,#assembly_accession,url
1122088,GCA_046498945.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/0...
1122089,GCA_046499025.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/0...
1122090,GCA_046499045.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/0...
1122091,GCA_046499055.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/0...
1122092,GCA_046499065.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/0...


## Step 6 — Save filtered metadata

The filtered metadata table is saved as a tab-separated file. This file serves as a reproducibility record of exactly which assemblies were identified and downloaded, including their accession numbers, submission dates, assembly status, and other fields from the original `assembly_summary`.

In [ ]:
meta_path = DATA_DIR / "ccg_unam_metadata.tsv"
ccg_df.drop(columns=["submitter_clean"]).to_csv(meta_path, sep="\t", index=False)
print(f"Metadata saved to: {meta_path}")

Metadata saved to: c:\Users\hayde\Downloads\ncbi-ccg-unam-s-aureus-retrieval\output\data\ccg_unam_metadata.tsv


## Step 7 — Download genomes

Each assembly is downloaded as a gzip-compressed FASTA file (`.fna.gz`), decompressed to an uncompressed FASTA file (`.fna`), and the compressed copy is deleted to save disk space.

The loop is designed to be safely re-run: files that already exist locally are skipped automatically, and any incomplete `.gz` files caused by a failed download are removed before the next attempt.

> **Note:** Downloading 108 genomes may take 20–60 minutes depending on your connection speed and NCBI server load.

In [16]:
def download_file(url: str, dest: Path) -> None:
    """Stream-download *url* to *dest*, raising an error on HTTP failure."""
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)


def decompress_gz(gz_path: Path, out_path: Path) -> None:
    """Decompress *gz_path* to *out_path* and remove the compressed file."""
    with gzip.open(gz_path, "rb") as f_in, open(out_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    gz_path.unlink()    # delete the .gz to save disk space


skipped = downloaded = errors = 0

for _, row in ccg_df.iterrows():
    url = row["url"]
    if not url:
        print(f"  [skip] {row['#assembly_accession']}: no URL.")
        skipped += 1
        continue

    gz  = GENOMES_DIR / (row["#assembly_accession"] + ".fna.gz")
    fna = GENOMES_DIR / row["file"]

    if fna.exists():
        print(f"  [skip] {row['#assembly_accession']}: already exists.")
        skipped += 1
        continue

    try:
        if not gz.exists():
            download_file(url, gz)
        decompress_gz(gz, fna)
        print(f"  [ok]   {row['#assembly_accession']}")
        downloaded += 1
    except Exception as exc:
        print(f"  [error] {row['#assembly_accession']}: {exc}")
        if gz.exists():
            gz.unlink()     # remove incomplete download
        errors += 1

print(f"\nDownloaded: {downloaded} | Skipped: {skipped} | Errors: {errors}")

  [ok]   GCA_046498945.1
  [ok]   GCA_046499025.1
  [ok]   GCA_046499045.1
  [ok]   GCA_046499055.1
  [ok]   GCA_046499065.1
  [ok]   GCA_046499105.1
  [ok]   GCA_046499125.1
  [ok]   GCA_046499145.1
  [ok]   GCA_046499155.1
  [ok]   GCA_046499185.1
  [ok]   GCA_046499205.1
  [ok]   GCA_046499225.1
  [ok]   GCA_046499245.1
  [ok]   GCA_046499265.1
  [ok]   GCA_046499275.1
  [ok]   GCA_046499305.1
  [ok]   GCA_046499325.1
  [ok]   GCA_046499335.1
  [ok]   GCA_046499365.1
  [ok]   GCA_046499385.1
  [ok]   GCA_046499405.1
  [ok]   GCA_046499415.1
  [ok]   GCA_046499445.1
  [ok]   GCA_046499455.1
  [ok]   GCA_046499475.1
  [ok]   GCA_046499505.1
  [ok]   GCA_046499525.1
  [ok]   GCA_046499545.1
  [ok]   GCA_046499555.1
  [ok]   GCA_046499565.1
  [ok]   GCA_046499585.1
  [ok]   GCA_046499625.1
  [ok]   GCA_046499635.1
  [ok]   GCA_046499655.1
  [ok]   GCA_046499665.1
  [ok]   GCA_046499705.1
  [ok]   GCA_046499715.1
  [ok]   GCA_046499745.1
  [ok]   GCA_046499755.1
  [ok]   GCA_046499765.1


## Final check

The number of `.fna` files in the output directory is verified against the number of records in the filtered metadata table.

In [17]:
fna_files = sorted(GENOMES_DIR.glob("*.fna"))
print(f".fna files in {GENOMES_DIR}: {len(fna_files)}")
for f in fna_files:
    print(f"  {f.name}")

.fna files in c:\Users\hayde\Downloads\ncbi-ccg-unam-s-aureus-retrieval\output\genomes: 108
  GCA_046498945.1.fna
  GCA_046499025.1.fna
  GCA_046499045.1.fna
  GCA_046499055.1.fna
  GCA_046499065.1.fna
  GCA_046499105.1.fna
  GCA_046499125.1.fna
  GCA_046499145.1.fna
  GCA_046499155.1.fna
  GCA_046499185.1.fna
  GCA_046499205.1.fna
  GCA_046499225.1.fna
  GCA_046499245.1.fna
  GCA_046499265.1.fna
  GCA_046499275.1.fna
  GCA_046499305.1.fna
  GCA_046499325.1.fna
  GCA_046499335.1.fna
  GCA_046499365.1.fna
  GCA_046499385.1.fna
  GCA_046499405.1.fna
  GCA_046499415.1.fna
  GCA_046499445.1.fna
  GCA_046499455.1.fna
  GCA_046499475.1.fna
  GCA_046499505.1.fna
  GCA_046499525.1.fna
  GCA_046499545.1.fna
  GCA_046499555.1.fna
  GCA_046499565.1.fna
  GCA_046499585.1.fna
  GCA_046499625.1.fna
  GCA_046499635.1.fna
  GCA_046499655.1.fna
  GCA_046499665.1.fna
  GCA_046499705.1.fna
  GCA_046499715.1.fna
  GCA_046499745.1.fna
  GCA_046499755.1.fna
  GCA_046499765.1.fna
  GCA_046499805.1.fna
  GCA_

## Summary

This workflow successfully retrieved **108 *Staphylococcus aureus* genome assemblies** associated with CCG-UNAM from NCBI GenBank.

| Step | Result |
|---|---|
| Total bacterial assemblies in GenBank | 3,026,217 |
| *S. aureus* assemblies | 122,614 |
| CCG-UNAM assemblies identified | 108 |
| Genomes downloaded successfully | 108 |
| Errors | 0 |

### Reproducibility

Results reflect the state of NCBI GenBank at the time of download. Re-running this notebook at a later date may return additional assemblies if new records have been deposited by CCG-UNAM. The filtered metadata file (`ccg_unam_metadata.tsv`) documents the exact set of assemblies retrieved in this run.